# SonarVision - Video Prediction

Runs the trained YOLOv8n model (`best.pt`) on videos placed in `input/video/`.

Pipeline per frame:
1. **Noise filtering** - the same `filter_noise` used across the project (median -> bilateral -> CLAHE)
2. **Prediction** - YOLOv8n on the cleaned frame
3. **Save-on-detect** - only frames with objects above the confidence threshold are saved (boxed) to `output/predictions/video_prediction/`
4. **Noise-filtered video** - a cleaned copy of each video is written to `output/noise_filter/input_noise_filter/videos/`

Input: `input/video/` (put your `.mp4` / `.avi` / `.mov` / `.mkv` / `.m4v` files there).


## 1. Setup

In [1]:
from pathlib import Path
import json
import colorsys

import cv2
import numpy as np
from ultralytics import YOLO

ROOT = Path(r"D:\1. Project Program\1.SIH\SonarVision")
BEST = None
for _c in (ROOT / "backend" / "best" / "best.pt", ROOT / "backend" / "best.pt"):
    if _c.exists():
        BEST = _c
        break

def gen_colors(n):
    return {i: tuple(int(c * 255) for c in colorsys.hsv_to_rgb(i / max(n, 1), 0.8, 0.95)[::-1]) for i in range(n)}

CLASS_NAMES = {}
CLASS_COLORS = {}

def draw_boxes(img, boxes, label_map, class_colors, border_thick=3, text_scale=0.75, text_thick=2):
    """Draw each detection with its class color: thicker border + larger label above the box."""
    if boxes is None:
        return img
    for b in boxes:
        x1, y1, x2, y2 = map(int, b.xyxy[0])
        x1, y1 = max(x1, 0), max(y1, 0)
        x2, y2 = min(x2, img.shape[1] - 1), min(y2, img.shape[0] - 1)
        if x2 <= x1 or y2 <= y1:
            continue
        cls_id = int(b.cls[0])
        c = class_colors.get(cls_id, (0, 255, 0))
        cv2.rectangle(img, (x1, y1), (x2, y2), c, border_thick)
        lbl = "{} {:.0f}%".format(label_map.get(cls_id, cls_id), float(b.conf[0]) * 100)
        cv2.putText(img, lbl, (x1, max(y1 - 8, 15)),
                    cv2.FONT_HERSHEY_SIMPLEX, text_scale, c, text_thick)
    return img

NF_NB = ROOT / "backend" / "noise_filtering.ipynb"

CONF_THRESHOLD = 0.50   # only save frames with boxes above this confidence

VIDEO_DIR = ROOT / "input" / "video"
SAVE_DIR = ROOT / "output" / "predictions" / "video_prediction"
NF_VIDEO_DIR = ROOT / "output" / "noise_filter" / "input_noise_filter" / "videos"

VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".m4v"}


## 2. Load model + noise filter

Reuses `filter_noise` straight out of `noise_filtering.ipynb` (same loader trick as the other notebooks).

In [2]:
def load_noise_filter_from_nb(nb_path: Path):
    nb = json.loads(nb_path.read_text(encoding="utf-8"))
    ns = {"__name__": "noise_filtering_mod", "cv2": cv2, "np": np, "Path": Path}
    for cell in nb["cells"]:
        if cell.get("cell_type") != "code":
            continue
        source = "".join(cell.get("source", []))
        if "def filter_noise" in source or "def preprocess_for_model" in source:
            exec(source, ns)
    return ns


fns = load_noise_filter_from_nb(NF_NB)
filter_noise = fns["filter_noise"]

if BEST is not None:
    model = YOLO(str(BEST))
    CLASS_NAMES = model.names
    CLASS_COLORS = gen_colors(len(CLASS_NAMES))
    print("Model loaded:", BEST)
    print("Classes:", CLASS_NAMES)
else:
    model = None
    print("WARNING: best.pt not found - run model.ipynb first.")
print("Noise filter loaded from:", NF_NB)


Model loaded: D:\1. Project Program\1.SIH\SonarVision\backend\best\best.pt
Classes: {0: 'pipe', 1: 'shipwrecks', 2: 'cylinder', 3: 'ghostnet', 4: 'plane', 5: 'human'}
Noise filter loaded from: D:\1. Project Program\1.SIH\SonarVision\backend\noise_filtering.ipynb


## 3. Run video prediction

Each processed frame: noise-filter -> predict. A cleaned copy of the whole video is written to `output/noise_filter/input_noise_filter/videos/`, and **only** frames that detect an object (above `CONF_THRESHOLD`) are saved as boxed images to `output/predictions/video_prediction/`.

In [3]:
VIDEO_DIR.mkdir(parents=True, exist_ok=True)
SAVE_DIR.mkdir(parents=True, exist_ok=True)
NF_VIDEO_DIR.mkdir(parents=True, exist_ok=True)

def process_video(video_path: Path):
    stem = video_path.stem
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"  !! cannot open video: {video_path.name}")
        return 0, 0

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    nf_path = NF_VIDEO_DIR / f"{stem}.mp4"
    writer = cv2.VideoWriter(str(nf_path), fourcc, fps, (width, height))

    processed = 0
    saved = 0
    frame_id = 0

    try:
        while True:
            ok, raw = cap.read()
            if not ok:
                break
            frame_id += 1

            # 1) Apply the SAME noise filter as the image/realtime pipeline
            #    (median -> bilateral -> CLAHE). Filter only; YOLO letterboxes internally.
            clean = filter_noise(raw)

            # 2) Detection
            result = model(clean, conf=CONF_THRESHOLD, verbose=False)[0]
            boxes = result.boxes
            det = boxes is not None and len(boxes) > 0

            # 3) Write the noise-filtered frame into the cleaned video copy
            writer.write(clean)
            processed += 1

            # 4) Save ONLY frames that have detections above the confidence threshold (boxed)
            if det:
                saved += 1
                annotated = draw_boxes(clean.copy(), boxes, CLASS_NAMES, CLASS_COLORS)
                out_p = SAVE_DIR / f"{stem}_frame_{frame_id:05d}.png"
                cv2.imwrite(str(out_p), annotated)
                dets = [
                    f"{CLASS_NAMES.get(int(b.cls[0]), int(b.cls[0]))} {float(b.conf[0]):.2f}"
                    for b in boxes
                ]
                print(f"  saved {out_p.name}  ->  {dets}")
    finally:
        cap.release()
        writer.release()

    print(f"  {video_path.name}: {processed} frame(s) processed, {saved} detection frame(s) saved")
    print(f"  noise-filtered video -> {nf_path}")
    return processed, saved


totals = {}
videos = sorted(p for p in VIDEO_DIR.iterdir() if p.is_file() and p.suffix.lower() in VIDEO_EXTS)
if not videos:
    print(f"No videos found in {VIDEO_DIR}. Put your video files there and re-run.")

for v in videos:
    print(f"\n=== {v.name} ===")
    p, s = process_video(v)
    totals[v.name] = {"frames": p, "detections": s}

print("\n=== SUMMARY ===")
for name, t in totals.items():
    print(f"  {name}: {t['frames']} frame(s), {t['detections']} detection frame(s) saved")
print(f"Detection frames -> {SAVE_DIR}")
print(f"Noise-filtered videos -> {NF_VIDEO_DIR}")



=== Side Scan Sonar.mp4 ===
  saved Side Scan Sonar_frame_00154.png  ->  ['shipwrecks 0.94']
  saved Side Scan Sonar_frame_00155.png  ->  ['shipwrecks 0.96']
  saved Side Scan Sonar_frame_00156.png  ->  ['shipwrecks 0.97']
  saved Side Scan Sonar_frame_00157.png  ->  ['shipwrecks 0.97']
  saved Side Scan Sonar_frame_00158.png  ->  ['shipwrecks 0.97']
  saved Side Scan Sonar_frame_00159.png  ->  ['shipwrecks 0.95']
  saved Side Scan Sonar_frame_00160.png  ->  ['shipwrecks 0.95']
  saved Side Scan Sonar_frame_00161.png  ->  ['shipwrecks 0.95']
  saved Side Scan Sonar_frame_00162.png  ->  ['shipwrecks 0.87']
  saved Side Scan Sonar_frame_00167.png  ->  ['shipwrecks 0.72']
  saved Side Scan Sonar_frame_00168.png  ->  ['shipwrecks 0.73']
  saved Side Scan Sonar_frame_00169.png  ->  ['shipwrecks 0.62']
  saved Side Scan Sonar_frame_00170.png  ->  ['shipwrecks 0.62']
  saved Side Scan Sonar_frame_00171.png  ->  ['shipwrecks 0.61']
  saved Side Scan Sonar_frame_00172.png  ->  ['shipwrecks 0.5

> Tip: to re-run on new videos, place the files in `input/video/` and run cell 3 again. Already-saved detection frames keep their file names (video-based), so re-runs won't silently overwrite results from other videos.